In [1]:
import os
import requests

In [2]:
if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response= requests.get(url, timeout=30)
    response.raise_for_status()
    print(response.content[:50])
    with open(file_path, "wb") as f:
        f.write(response.content)

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [4]:
import re

In [5]:
text = "Hello, world. This, is a test."
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item for item in result if item.strip()]
result

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']

In [7]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
preprocessed[:30]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius',
 '--',
 'though',
 'a',
 'good',
 'fellow',
 'enough',
 '--',
 'so',
 'it',
 'was',
 'no',
 'great',
 'surprise',
 'to',
 'me',
 'to',
 'hear',
 'that',
 ',',
 'in']

In [8]:
len(preprocessed)

4690

2.3 Converting tokens into token IDs

In [16]:
all_words = sorted(set(preprocessed))
all_words[:10]

['!', '"', "'", '(', ')', ',', '--', '.', ':', ';']

In [19]:
vocab_size = len(all_words)
vocab_size

1130

In [25]:
vocab = {token: integer for integer, token in enumerate(all_words)}
vocab.__len__()

1130

In [32]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 10:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)


In [40]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[id] for id in ids])
        # replace incorrect formatting of punctuations ("hello , there" -> "hello, there")
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [41]:
tokenizer = SimpleTokenizerV1(vocab)

In [42]:
text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""

In [49]:
ids = tokenizer.encode(text)
ids[:5]

[1, 56, 2, 850, 988]

In [50]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [51]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

2.4 Adding special context tokens

In [53]:
tokenizer = SimpleTokenizerV1(vocab)
text = "Hello, do you like tea. Is this-- a test?"
tokenizer.encode(text)

KeyError: 'Hello'

In [65]:
all_tokens = sorted(list(set(preprocessed)))
len(all_tokens)
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}
len(vocab)

1132

In [66]:
for item in list(vocab.items())[-5:]:
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [68]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]

        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[id] for id in ids])
        # replace incorrect formatting of punctuations ("hello , there" -> "hello, there")
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [69]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

In [70]:
text

'Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.'

In [71]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [72]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

2.5 BytePair encoding

In [73]:
# tiktoken pip library used

In [76]:
import importlib
import tiktoken

In [79]:
print("tiktoken version: ", importlib.metadata.version("tiktoken"))
tiktoken.__version__

tiktoken version:  0.14.0


'0.14.0'

In [80]:
tokenizer = tiktoken.get_encoding("gpt2")

In [82]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

In [86]:
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
integers[:10]

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554]

In [88]:
strings = tokenizer.decode(integers)
strings

'Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.'

2.6 Data sampling with a sliding window

In [89]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [93]:
enc_text = tokenizer.encode(raw_text)

In [94]:
len(enc_text)

5145

In [97]:
enc_sample = enc_text[50:]

In [98]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]

In [101]:
print(x)
y

[290, 4920, 2241, 287]


[4920, 2241, 287, 257]

In [103]:
# one-by-one prediction
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [105]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a
